# Granite docling 258M

### Using the transformers library

In [1]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


Start by initializing the model and processor.

In [2]:
from transformers import AutoProcessor, AutoModelForImageTextToText

model_name = "ibm-granite/granite-docling-258M"
granite_docling_processor = AutoProcessor.from_pretrained(model_name)
granite_docling_model = AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path=model_name,
    # dtype=torch.bfloat16,
).to(device)  # type: ignore

Prepare the input to be passed to the model.

In [3]:
from PIL import Image
from transformers.image_utils import load_image


# Provide a URL or a PIL image
image_path = "dataset/invoice.png"
image = Image.open(image_path)
image = load_image(image=image_path)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling."},
        ],
    },
]

prompt = granite_docling_processor.apply_chat_template(
    conversation=messages, add_generation_prompt=True
)
inputs = granite_docling_processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(device)

Generate the output, which will contain the extracted text. Even with a GPU, it took around 5 minutes to process just one image. The `doc_tags` variable contains the raw output from the model.

**Edit:** I found a solution to the slowness issue. You need to explicitely set `use_cache` to `True` to use KV caching.

In [4]:
generated_ids = granite_docling_model.generate(
    **inputs, max_new_tokens=8192, use_cache=True
)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doc_tags = granite_docling_processor.batch_decode(
    trimmed_generated_ids,
    skip_special_tokens=False,
)[0].lstrip()

In [5]:
print(doc_tags)

<doctag><section_header_level_1><loc_27><loc_25><loc_130><loc_52>RedmineCRM</section_header_level_1>
<text><loc_26><loc_53><loc_137><loc_112>Company representative name Your company address Tax ID phone: fax:</text>
<text><loc_298><loc_62><loc_340><loc_76>Invoice ID:</text>
<text><loc_344><loc_62><loc_409><loc_76>INV/20111209-22</text>
<text><loc_290><loc_82><loc_340><loc_95>Invoice date:</text>
<text><loc_351><loc_82><loc_409><loc_95>12/08/2011</text>
<text><loc_290><loc_101><loc_340><loc_115>Text custom field:</text>
<text><loc_351><loc_101><loc_415><loc_115>Visible field in PDF</text>
<text><loc_314><loc_120><loc_340><loc_134>Bill to:</text>
<text><loc_351><loc_120><loc_432><loc_134>"Romashka" Ltd.</text>
<text><loc_314><loc_138><loc_340><loc_152>1600</text>
<text><loc_351><loc_138><loc_447><loc_152>Amphitheatre Parkway</text>
<text><loc_314><loc_155><loc_340><loc_169>Mountain</text>
<text><loc_351><loc_155><loc_414><loc_169>View, CA 94043</text>
<otsl><loc_27><loc_204><loc_474><loc

Render the output in markdown format.

In [6]:
from IPython.display import Markdown, display
from docling_core.types.doc.document import DoclingDocument, DocTagsDocument


doctag_document = DocTagsDocument.from_doctags_and_image_pairs(
    doctags=[doc_tags], images=[image]
)
docling_document = DoclingDocument.load_from_doctags(
    doctag_document=doctag_document, document_name="Document"
)
extracted_text_markdown = docling_document.export_to_markdown()
display(Markdown(extracted_text_markdown))

## RedmineCRM

Company representative name Your company address Tax ID phone: fax:

Invoice ID:

INV/20111209-22

Invoice date:

12/08/2011

Text custom field:

Visible field in PDF

Bill to:

"Romashka" Ltd.

1600

Amphitheatre Parkway

Mountain

View, CA 94043

|   # | Description                                                                                                               | Qty   | Units   |   Unit price (EUR) |   Total (EUR) |
|-----|---------------------------------------------------------------------------------------------------------------------------|-------|---------|--------------------|---------------|
|   1 | Projecting - Context menu for invoices list                                                                               | x1.0  | hours   |              50.00 |         50.00 |
|   2 | Develop - Invoice number format template - [PRO] Duplicating invoices - Language support - Context menu for invoices list | x17.0 | hours   |              40.00 |        680.00 |
|   3 | Analysis - [PRO] Duplicating invoices - Language support                                                                  | x3.0  | hours   |              35.00 |        105.00 |

Sub total:

835.00

Discount (10.0%):

150.30

Total (EUR):

901.80

Your billing information (Bank, Address, IBAN, SWIFT &amp; etc.)

Was ist mit Beschreibung

Sub total:

835.00

Discount (10.0%):

150.30

Total (EUR):

901.80

Save the extracted text to a file so that we can review it later.

In [7]:
file_path = "granite_docling_output.txt"
with open(file_path, "w") as f:
    f.write(extracted_text_markdown)